In [1]:
"""
evaluate_recommender_topk_bq.py - Top-K evaluation of the RATING-AWARE recommender,
reading from and writing to BigQuery (no Excel).

INPUT (read-only):
  RECS  : Silver  cbf_user_recommendations_rating_aware_eval10   (the top-10 recs)
  TRAIN : Bronze  CBF_train_split_latest_80pct                    (train item universe)
  TEST  : Bronze  CBF_test_split_latest_20pct                     (held-out ground truth)

OUTPUT (written to Silver; each run REPLACES these eval-result tables only):
  cbf_rating_aware_recommender_evaluation_top10           - PER-USER metrics (main, detailed)
  cbf_rating_aware_recommender_evaluation_top10_summary   - OVERALL KPIs (one row)
  cbf_rating_aware_recommender_evaluation_top10_hits      - PER (user, held-out item) hit detail

ELIGIBILITY (unchanged from the original program): a test row is kept only if its
parent_asin appears in the TRAIN item universe (inner_join_test_to_train_items).
Test items never seen in training are excluded from evaluation.
"""
import os
import math
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import shutil

from google.colab import drive
from google.colab import auth
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

mount_point = '/content/drive'
if os.path.exists(mount_point):
    if os.path.ismount(mount_point):
        print(f"[{mount_point}] is currently mounted. Attempting to unmount.")
        try:
            drive.flush_and_unmount()
            print(f"[{mount_point}] unmounted successfully.")
        except Exception as e:
            print(f"Error unmounting [{mount_point}]: {e}")
    if os.path.isdir(mount_point) and os.listdir(mount_point):
        print(f"[{mount_point}] non-empty (or unmount failed). Clearing.")
        try:
            shutil.rmtree(mount_point); os.makedirs(mount_point)
            print(f"[{mount_point}] cleared and recreated.")
        except Exception as e:
            print(f"Error clearing [{mount_point}]: {e}")
    elif not os.path.isdir(mount_point) and os.path.exists(mount_point):
        os.remove(mount_point); os.makedirs(mount_point)

drive.mount(mount_point, force_remount=True)

sys.path.insert(0, '/content')
auth.authenticate_user()

import cbf_config as cfg
from cbf_common import Timer

# =========================================================
# CONSTANTS
# =========================================================
PROJECT = "hosted-s-nelsonmendenilla-680e"

RECS_TABLE  = f"{cfg.PROJECT}.EShop_Silver_Virtualization.cbf_user_recommendations_rating_aware_hybrid_eval20"
TRAIN_TABLE = f"{cfg.PROJECT}.EShop_Bronze_Datamarts.CBF_train_split_latest_80pct"
TEST_TABLE  = f"{cfg.PROJECT}.EShop_Bronze_Datamarts.CBF_test_split_latest_20pct"

OUTPUT_DATASET = f"{cfg.PROJECT}.EShop_Silver_Virtualization"
EVAL_TABLE         = f"{cfg.PROJECT}.{cfg.DATASET}.cbf_rating_aware_hybrid_recommender_evaluation_top20"
EVAL_SUMMARY_TABLE = f"{cfg.PROJECT}.{cfg.DATASET}.cbf_rating_aware_hybrid_recommender_evaluation_top20_summary"
EVAL_HITS_TABLE    = f"{cfg.PROJECT}.{cfg.DATASET}.cbf_rating_aware_hybrid_recommender_evaluation_top20_hits"

# Top-K cutoff used for evaluation.
TOP_K = 20

# ---------------------------------------------------------
# normalisation (same logic as the Excel version)
# ---------------------------------------------------------
def normalize_recommendations(df: pd.DataFrame, k: int) -> pd.DataFrame:
    required = {"user_id", "neighbour_item"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Recommendations is missing required columns: {sorted(missing)}")

    out = df.copy()
    out["user_id"] = out["user_id"].astype(str)
    out["neighbour_item"] = out["neighbour_item"].astype(str)

    has_rank = "recommendation_rank" in out.columns
    has_score = "pred_rating" in out.columns
    if has_rank:
        out["recommendation_rank"] = pd.to_numeric(out["recommendation_rank"], errors="coerce")
    if has_score:
        out["pred_rating"] = pd.to_numeric(out["pred_rating"], errors="coerce")

    if has_rank:
        out = (out.sort_values(["user_id", "recommendation_rank", "neighbour_item"],
                               ascending=[True, True, True])
                  .drop_duplicates(subset=["user_id", "neighbour_item"], keep="first")
                  .sort_values(["user_id", "recommendation_rank", "neighbour_item"],
                               ascending=[True, True, True])
                  .groupby("user_id", as_index=False, group_keys=False).head(k).copy())
    elif has_score:
        out = (out.sort_values(["user_id", "pred_rating", "neighbour_item"],
                               ascending=[True, False, True])
                  .drop_duplicates(subset=["user_id", "neighbour_item"], keep="first")
                  .sort_values(["user_id", "pred_rating", "neighbour_item"],
                               ascending=[True, False, True])
                  .groupby("user_id", as_index=False, group_keys=False).head(k).copy())
        out["recommendation_rank"] = out.groupby("user_id").cumcount() + 1
    else:
        out = (out.sort_values(["user_id", "neighbour_item"], ascending=[True, True])
                  .drop_duplicates(subset=["user_id", "neighbour_item"], keep="first")
                  .groupby("user_id", as_index=False, group_keys=False).head(k).copy())
        out["recommendation_rank"] = out.groupby("user_id").cumcount() + 1

    if "pred_rating" not in out.columns:
        out["pred_rating"] = np.nan

    return out[["user_id", "neighbour_item", "recommendation_rank", "pred_rating"]].copy()


def normalize_test(df: pd.DataFrame) -> pd.DataFrame:
    required = {"user_id", "parent_asin"}            # rating no longer required (threshold removed)
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Test is missing required columns: {sorted(missing)}")

    out = df.copy()
    out["user_id"] = out["user_id"].astype(str)
    out["parent_asin"] = out["parent_asin"].astype(str)
    if "rating" in out.columns:
        out["rating"] = pd.to_numeric(out["rating"], errors="coerce")
    out = out.dropna(subset=["user_id", "parent_asin"]).copy()
    return out


def normalize_train_items(df: pd.DataFrame) -> pd.DataFrame:
    required = {"parent_asin"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Train is missing required columns: {sorted(missing)}")
    out = df.copy()
    out["parent_asin"] = out["parent_asin"].astype(str)
    out = out.dropna(subset=["parent_asin"])[["parent_asin"]].drop_duplicates().copy()
    return out

def inner_join_test_to_train_items(test_df: pd.DataFrame,
                                   train_items_df: pd.DataFrame):
    """Keep only test rows whose parent_asin exists in the training item universe."""
    merged = test_df.merge(train_items_df[["parent_asin"]], on="parent_asin",
                           how="left", indicator=True)
    filtered = merged.loc[merged["_merge"] == "both", test_df.columns].copy()
    excluded = merged.loc[merged["_merge"] == "left_only", test_df.columns].copy()
    return filtered, excluded

# ---------------------------------------------------------
# evaluation
# ---------------------------------------------------------
def evaluate_topk(recs_df: pd.DataFrame, test_df: pd.DataFrame, k: int):
    # No threshold: every held-out test item is relevant.
    relevant_test = test_df.copy()

    rec_groups = {uid: grp.sort_values("recommendation_rank")
                  for uid, grp in recs_df.groupby("user_id")}
    rel_groups = {uid: grp.copy()
                  for uid, grp in relevant_test.groupby("user_id")}

    all_users = sorted(set(rel_groups.keys()) | set(rec_groups.keys()))
    rows = []
    hits_rows = []

    for user_id in all_users:
        rec_u = rec_groups.get(user_id)
        rel_u = rel_groups.get(user_id)

        rec_items, rank_map = [], {}
        if rec_u is not None and not rec_u.empty:
            rec_items = rec_u["neighbour_item"].astype(str).tolist()
            rank_map = dict(zip(rec_u["neighbour_item"].astype(str),
                                rec_u["recommendation_rank"].astype(int)))

        rel_items = set()
        if rel_u is not None and not rel_u.empty:
            rel_items = set(rel_u["parent_asin"].astype(str).tolist())

        hits = sorted(rel_items.intersection(rec_items),
                      key=lambda x: rank_map.get(x, 10**9))
        hit_count = len(hits)
        rel_count = len(rel_items)
        rec_count = len(rec_items)

        precision = hit_count / k if k > 0 else np.nan
        recall = hit_count / rel_count if rel_count > 0 else np.nan
        hit_rate = 1.0 if hit_count > 0 else (0.0 if rel_count > 0 else np.nan)

        dcg = 0.0
        for item in hits:
            dcg += 1.0 / np.log2(rank_map[item] + 1)
        ideal_hits = min(rel_count, k)
        idcg = sum(1.0 / np.log2(r + 1) for r in range(1, ideal_hits + 1))
        ndcg = (dcg / idcg) if rel_count > 0 and idcg > 0 else np.nan

        ap = np.nan
        if rel_count > 0:
            running_hits = 0
            precisions_at_hits = []
            for rank, item in enumerate(rec_items, start=1):
                if item in rel_items:
                    running_hits += 1
                    precisions_at_hits.append(running_hits / rank)
            denom = min(rel_count, k)
            ap = (sum(precisions_at_hits) / denom) if denom > 0 else np.nan

        rows.append({
            "user_id": user_id,
            "num_recommended_at_k": rec_count,
            "num_relevant_test_items": rel_count,
            "num_hits": hit_count,
            "hit_items": ", ".join(hits),
            "precision_at_k": precision,
            "recall_at_k": recall,
            "hit_rate_at_k": hit_rate,
            "ndcg_at_k": ndcg,
            "avg_precision_at_k": ap,
        })

        # per (user, held-out test item) detail - the lowest meaningful grain
        for item in rel_items:
            hits_rows.append({
                "user_id": user_id,
                "parent_asin": item,
                "is_hit": item in rec_items,
                "hit_rank": int(rank_map[item]) if item in rec_items else None,
            })

    per_user = pd.DataFrame(rows)
    hits_detail = pd.DataFrame(hits_rows)
    if not hits_detail.empty:
        hits_detail["is_hit"] = hits_detail["is_hit"].astype(bool)
        hits_detail["hit_rank"] = hits_detail["hit_rank"].astype("Int64")  # nullable int -> BQ INTEGER

    eval_mask = per_user["num_relevant_test_items"] > 0
    evaluated = per_user.loc[eval_mask].copy()

    summary = pd.DataFrame([{
        "k": k,
        "test_rows_after_exclusion": int(len(test_df)),
        "users_with_relevant_test_items": int(eval_mask.sum()),
        "users_in_recommendations": int(recs_df["user_id"].nunique()),
        "users_with_any_topk_recommendations": int((per_user["num_recommended_at_k"] > 0).sum()),
        "precision_at_k_mean": float(evaluated["precision_at_k"].mean()) if not evaluated.empty else np.nan,
        "recall_at_k_mean": float(evaluated["recall_at_k"].mean()) if not evaluated.empty else np.nan,
        "hit_rate_at_k_mean": float(evaluated["hit_rate_at_k"].mean()) if not evaluated.empty else np.nan,
        "ndcg_at_k_mean": float(evaluated["ndcg_at_k"].mean()) if not evaluated.empty else np.nan,
        "avg_precision_at_k_mean": float(evaluated["avg_precision_at_k"].mean()) if not evaluated.empty else np.nan,
    }])

    return summary, per_user, hits_detail


# ---------------------------------------------------------
# BigQuery I/O
# ---------------------------------------------------------
def _read_bq(client, sql: str) -> pd.DataFrame:
    return client.query(sql).result().to_dataframe()


def _write_bq(client, df: pd.DataFrame, table_id: str):
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    client.load_table_from_dataframe(df, table_id, job_config=job_config).result()
    print(f"  wrote {len(df):,} rows -> {table_id}")


def main():
    if bigquery is None:
        raise RuntimeError("google-cloud-bigquery is not available in this environment.")
    if auth is not None:
        auth.authenticate_user()

    client = bigquery.Client(project=PROJECT)

    print("Reading from BigQuery...")
    # recs: alias the eval10 columns to the names the evaluator expects
    recs_raw = _read_bq(client, f"""
        SELECT user_id,
               recommended_asin AS neighbour_item,
               `rank`           AS recommendation_rank,
               predicted_rating AS pred_rating
        FROM `{RECS_TABLE}`
    """)
    # train item universe (DISTINCT in SQL - tiny, avoids pulling all train rows)
    train_raw = _read_bq(client, f"""
        SELECT DISTINCT parent_asin
        FROM `{TRAIN_TABLE}`
        WHERE parent_asin IS NOT NULL
    """)
    # held-out ground truth
    test_raw = _read_bq(client, f"""
        SELECT user_id, parent_asin, rating
        FROM `{TEST_TABLE}`
        WHERE user_id IS NOT NULL AND parent_asin IS NOT NULL
    """)

    recs = normalize_recommendations(recs_raw, TOP_K)
    train_items = normalize_train_items(train_raw)
    test_all = normalize_test(test_raw)
    #test_filtered, excluded_rows = inner_join_test_to_train_items(test_all, train_items)

    #summary, per_user, hits_detail = evaluate_topk(recs, test_filtered, TOP_K)
    summary, per_user, hits_detail = evaluate_topk(recs, test_all, TOP_K)

    # same diagnostic counts the Excel version surfaced, folded into the summary
    summary.insert(1, "train_unique_items", len(train_items))
    summary.insert(2, "test_rows_original", len(test_all))
    #summary.insert(3, "excluded_rows_not_in_train", len(excluded_rows))

    print("\nWriting results to BigQuery...")
    _write_bq(client, per_user, EVAL_TABLE)
    _write_bq(client, summary, EVAL_SUMMARY_TABLE)
    _write_bq(client, hits_detail, EVAL_HITS_TABLE)

    print("\nEvaluation complete.")
    print("\nOverall KPI summary:")
    print(summary.to_string(index=False))


if __name__ == "__main__":
    main()


Mounted at /content/drive
Reading from BigQuery...

Writing results to BigQuery...
  wrote 18,959 rows -> hosted-s-nelsonmendenilla-680e.EShop_Silver_Virtualization.cbf_rating_aware_hybrid_recommender_evaluation_top20
  wrote 1 rows -> hosted-s-nelsonmendenilla-680e.EShop_Silver_Virtualization.cbf_rating_aware_hybrid_recommender_evaluation_top20_summary
  wrote 1,354 rows -> hosted-s-nelsonmendenilla-680e.EShop_Silver_Virtualization.cbf_rating_aware_hybrid_recommender_evaluation_top20_hits

Evaluation complete.

Overall KPI summary:
 k  train_unique_items  test_rows_original  test_rows_after_exclusion  users_with_relevant_test_items  users_in_recommendations  users_with_any_topk_recommendations  precision_at_k_mean  recall_at_k_mean  hit_rate_at_k_mean  ndcg_at_k_mean  avg_precision_at_k_mean
20               17170                1362                       1362                             687                     18959                                18959             0.002984          0